# **Homework 3: Multiple Minimum Support**

Deadline: 5th March 2026

Name: Aleena Zahra 23i-2514 DS-B

## Overview: 
The objective of this homework is to perform Market Basket Analysis using the Multiple Minimum Item Support (MMIS) algorithm and extract association rules from transactional data.
Unlike standard Apriori (single global minimum support), MMIS allows different minimum support values for different items, enabling discovery of rules involving rare but important products.

Dataset: 
You are given a dataset titled online_retail.csv. 


In [1]:

import pandas as pd
import numpy as np
from itertools import combinations
from collections import defaultdict



In [2]:
df = pd.read_excel('Aleena Zahra - online_retail.xlsx')


In [3]:
df.head()

,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom



## **Tasks**

## 1. Transaction Construction 

● Construct transactions such that each invoice corresponds to one basket. 

● Represent each basket as a list of purchased items. 

● Remove cancelled invoices and missing descriptions.



In [4]:
# remove cancelled transactions
df = df[df['Quantity'] > 0]
# remove null values 
df = df.dropna(subset=['Description', 'InvoiceDate', 'Quantity'])

# drop unit price,country and quantity columns
df = df.drop(columns=['UnitPrice', 'Country', 'Quantity'])


In [5]:

# lowercase description column because it bothers me
df['Description'] = df['Description'].str.lower()

In [6]:
# get the number of unique invoiceDates
print(df['InvoiceDate'].nunique())



32


In [7]:
baskets = (
    df.groupby('InvoiceDate')['Description']
      .apply(list)
      .reset_index(name='items')
)
print(len(baskets))

32



## 2. Time Based Baskets 
● Extract hour from InvoiceDate 

● Divide transactions into 4 groups: Morning, Afternoon, Evening, Night 

● Create 4 separate basket collections based on these time groups. 


In [8]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Hour'] = df['InvoiceDate'].dt.hour

# Define time groups
def get_time_period(hour):
    if 5 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

df['TimePeriod'] = df['Hour'].apply(get_time_period)

In [9]:
baskets_by_time = {}

for period in ['Morning', 'Afternoon', 'Evening', 'Night']:
    subset = df[df['TimePeriod'] == period]
    baskets_by_time[period] = (
        subset.groupby('InvoiceDate')['Description']
              .apply(list)
              .tolist()                     # list of lists
    )
    
    print(f"{period}: {len(baskets_by_time[period])} transactions")

Morning: 32 transactions
Afternoon: 0 transactions
Evening: 0 transactions
Night: 0 transactions



## 3. One Hot Encoding 
Use a library-based encoder  that converts a list of baskets into a one hot encoded Dataframe.



In [10]:
def one_hot_encode(baskets: list[list[str]]) -> pd.DataFrame:

    from sklearn.preprocessing import MultiLabelBinarizer
    mlb = MultiLabelBinarizer()
    ohe_array = mlb.fit_transform(baskets)
    return pd.DataFrame(ohe_array, columns=mlb.classes_)



## 4.  Multiple Minimum Item Support (MMIS)

### Step 1: Assign Item-Specific Supports
Create a MIS dictionary where:
        **MIS(i)=max(β×support(i),LS)**

β = scaling factor (example: 0.5)


LS = least support threshold (example: 0.01)


support(i) = individual item support

-> Compute item supports
-> Generate MIS values dynamically (no hardcoding)


In [11]:

def compute_item_supports(baskets: list[list[str]]) -> dict[str, float]:
    n = len(baskets)
    counts: dict[str, int] = defaultdict(int)
    for basket in baskets:
        for item in set(basket):         
            counts[item] += 1
    return {item: count / n for item, count in counts.items()}


In [12]:


def build_mis_dict(item_supports: dict[str, float],
                   beta: float,
                   ls: float) -> dict[str, float]:
    return {item: max(beta * sup, ls) for item, sup in item_supports.items()}




### Step 2: Implement MMIS Algorithm



In [18]:

def mmis_frequent_itemsets(baskets: list[list[str]],
                            beta: float,
                            ls: float) -> list[tuple[frozenset, float]]:
    n = len(baskets)
    if n == 0:
        return []
    basket_sets = [frozenset(b) for b in baskets]
    item_supports = compute_item_supports(baskets)
    mis_dict      = build_mis_dict(item_supports, beta, ls)
    sorted_items = sorted(item_supports.keys(), key=lambda x: mis_dict[x])

    def support_of(itemset: frozenset) -> float:
        return sum(1 for b in basket_sets if itemset <= b) / n

    def min_mis(itemset: frozenset) -> float:
        return min(mis_dict[i] for i in itemset)

    frequent_1: list[tuple[frozenset, float]] = []
    seed: list[str] = []         

    first_item_mis = mis_dict[sorted_items[0]]

    for item in sorted_items:
        sup = item_supports[item]
        if sup >= mis_dict[item]:         
            frequent_1.append((frozenset([item]), sup))
        if sup >= first_item_mis:           
            seed.append(item)

    all_frequent: list[tuple[frozenset, float]] = list(frequent_1)

    prev_frequent = [fs for fs, _ in frequent_1]

    k = 2
    while prev_frequent and k<= 3:
        candidates: list[frozenset] = []

        sorted_prev = [sorted(list(fs), key=lambda x: mis_dict[x])
                       for fs in prev_frequent]

        for i in range(len(sorted_prev)):
            for j in range(i + 1, len(sorted_prev)):
                a, b = sorted_prev[i], sorted_prev[j]
                if a[:-1] == b[:-1] and a[-1] != b[-1]:
                    candidate = frozenset(a) | frozenset(b)

                    min_mis_item = min(candidate, key=lambda x: mis_dict[x])
                    pruned = False
                    for subset in combinations(candidate, k - 1):
                        subset_fs = frozenset(subset)
                        if min_mis_item not in subset_fs:
                            continue       
                        if subset_fs not in prev_frequent:
                            pruned = True
                            break

                    if not pruned and candidate not in candidates:
                        candidates.append(candidate)

        new_frequent: list[frozenset] = []
        for cand in candidates:
            sup = support_of(cand)
            if sup >= min_mis(cand):
                new_frequent.append(cand)
                all_frequent.append((cand, sup))

        prev_frequent = new_frequent
        k += 1

    return all_frequent



## 5.  Experiment with Two Parameter Settings


In [20]:

EXPERIMENTS = [
    {"name": "Exp1", "beta": 0.5, "ls": 0.1},
    {"name": "Exp2", "beta": 0.7, "ls": 0.2}
]

PERIODS = ["Morning"]

experiment_results: dict = {}   


for exp in EXPERIMENTS:

    print(f"  {exp['name']}  |  β={exp['beta']}  |  LS={exp['ls']}")


    for period in PERIODS:
        baskets = baskets_by_time[period]
        frequent = mmis_frequent_itemsets(baskets, exp["beta"], exp["ls"])
        experiment_results[(exp["name"], period)] = frequent

        # Stats
        num_frequent = len(frequent)
        largest_k    = max((len(fs) for fs, _ in frequent), default=0)

        # Top-5 by support
        top5 = sorted(frequent, key=lambda x: x[1], reverse=True)[:5]

        print(f"\n  [{period}]")
        print(f"    Frequent itemsets : {num_frequent}")
        print(f"    Largest itemset   : {largest_k} items")
        print(f"    Top-5 by support  :")
        for fs, sup in top5:
            items_str = ", ".join(sorted(fs))
            print(f"      sup={sup:.4f}  →  {{{items_str}}}")



  Exp1  |  β=0.5  |  LS=0.01

  [Morning]
    Frequent itemsets : 61817
    Largest itemset   : 3 items
    Top-5 by support  :
      sup=0.2188  →  {white hanging heart t-light holder}
      sup=0.1562  →  {set 7 babushka nesting boxes}
      sup=0.1562  →  {knitted union flag hot water bottle}
      sup=0.1562  →  {hand warmer union jack}
      sup=0.1562  →  {knitted union flag hot water bottle, white hanging heart t-light holder}
  Exp2  |  β=0.7  |  LS=0.02

  [Morning]
    Frequent itemsets : 60436
    Largest itemset   : 3 items
    Top-5 by support  :
      sup=0.2188  →  {white hanging heart t-light holder}
      sup=0.1562  →  {set 7 babushka nesting boxes}
      sup=0.1562  →  {knitted union flag hot water bottle}
      sup=0.1562  →  {hand warmer union jack}
      sup=0.1562  →  {knitted union flag hot water bottle, white hanging heart t-light holder}


## 6. Association Rule Mining


In [ ]:

def generate_rules(frequent_itemsets: list[tuple[frozenset, float]],
                   baskets: list[list[str]],
                   min_confidence: float = 0.6,
                   min_lift: float = 1.0) -> list[dict]:
    n = len(baskets)
    basket_sets = [frozenset(b) for b in baskets]

    sup_lookup: dict[frozenset, float] = {fs: sup for fs, sup in frequent_itemsets}

    def support_of(itemset: frozenset) -> float:
        cached = sup_lookup.get(itemset)
        if cached is not None:
            return cached
        return sum(1 for b in basket_sets if itemset <= b) / n

    rules: list[dict] = []

    for itemset, itemset_sup in frequent_itemsets:
        if len(itemset) < 2:
            continue
        items = list(itemset)
        for r in range(1, len(items)):
            for antecedent in combinations(items, r):
                antecedent_fs  = frozenset(antecedent)
                consequent_fs  = itemset - antecedent_fs

                ant_sup  = support_of(antecedent_fs)
                cons_sup = support_of(consequent_fs)

                if ant_sup == 0:
                    continue

                confidence = itemset_sup / ant_sup
                lift       = confidence / cons_sup if cons_sup > 0 else 0

                if confidence >= min_confidence and lift > min_lift:
                    rules.append({
                        "antecedent" : antecedent_fs,
                        "consequent" : consequent_fs,
                        "support"    : itemset_sup,
                        "confidence" : confidence,
                        "lift"       : lift,
                    })

    return rules


all_rules: dict = {}   

for exp in EXPERIMENTS:
    print(f"  {exp['name']}  |  β={exp['beta']}  |  LS={exp['ls']}")
   

    for period in PERIODS:
        baskets   = baskets_by_time[period]
        frequent  = experiment_results[(exp["name"], period)]
        rules     = generate_rules(frequent, baskets,
                                   min_confidence=0.6, min_lift=1.0)
        all_rules[(exp["name"], period)] = rules

        top5_rules = sorted(rules, key=lambda r: r["lift"], reverse=True)[:5]

        print(f"\n  [{period}]  —  {len(rules)} rules generated")
        print(f"  Top-5 by Lift:")
        if not top5_rules:
            print("    (no rules)")
        for r in top5_rules:
            ant = ", ".join(sorted(r["antecedent"]))
            con = ", ".join(sorted(r["consequent"]))
            print(f"{{{ant}}} → {{{con}}}")
            print(f"sup={r['support']:.4f}  conf={r['confidence']:.4f}  lift={r['lift']:.4f}")



  Exp1  |  β=0.5  |  LS=0.01

  [Morning]  —  307628 rules generated
  Top-5 by Lift:
    {red coat rack paris fashion} → {yellow coat rack paris fashion}
      sup=0.0312  conf=1.0000  lift=32.0000
    {yellow coat rack paris fashion} → {red coat rack paris fashion}
      sup=0.0312  conf=1.0000  lift=32.0000
    {red coat rack paris fashion} → {recipe box with metal heart}
      sup=0.0312  conf=1.0000  lift=32.0000
    {recipe box with metal heart} → {red coat rack paris fashion}
      sup=0.0312  conf=1.0000  lift=32.0000
    {red coat rack paris fashion} → {blue coat rack paris fashion}
      sup=0.0312  conf=1.0000  lift=32.0000
  Exp2  |  β=0.7  |  LS=0.02

  [Morning]  —  304909 rules generated
  Top-5 by Lift:
    {red coat rack paris fashion} → {yellow coat rack paris fashion}
      sup=0.0312  conf=1.0000  lift=32.0000
    {yellow coat rack paris fashion} → {red coat rack paris fashion}
      sup=0.0312  conf=1.0000  lift=32.0000
    {red coat rack paris fashion} → {recipe b

## 7. Comparative Analysis


Answer the following:

      Which time period produces the strongest rules (highest Lift)?
Morning as it is the only one with any transactions.

      Which item combinations appear in multiple time periods?
none

      How does increasing β affect: 
      Number of frequent itemsets?
It decreases them since rare and frequent items are less often paired up so there are less number of frequent itemsets. 

      Number of rules?
decrease as the number of itemsets decrease.


      Does MMIS discover rare-item rules that standard Apriori might miss? Explain with examples.
      
yes because it shows rare items with other rare items as a combination. for example 

glass star frosted t-light holder} → {white metal lantern}
sup=0.1250  conf=1.0000  lift=8.0000

this is present rarely in the dataset being 12% in transactions but every time both items show up together so this rule is found in MMIS but not in apriori assuming minsup> 0.12 is set.